# 05 — Organize Clean Data
# Giai đoạn 1 — Mục 1.0 — Tổ chức 40 file sạch vào data/clean/

- **Đầu vào**: `outputs/tables/manifest_filtered.csv` (từ notebook 01)
- **Đầu ra**:
  - `data/clean/` — 40 file sạch
  - `outputs/tables/manifest_clean.csv` — manifest kèm cột `fs` (sampling rate thật)

**Cập nhật:** cột `fs` giờ lấy TRỰC TIẾP từ `resolved_sample_rate_hz` (đã được
`io_utils.run_sanity_checks()` xác định ở notebook 01 dựa trên thời lượng tín
hiệu thực đo — có căn cứ, không đoán mò). KHÔNG còn dict gõ tay
`NORMAL_FS_OVERRIDE` — nguồn duy nhất cho fs của mọi file (kể cả 4 file
Normal) giờ là `resolved_sample_rate_hz`, tránh 2 nguồn dữ liệu chênh nhau
âm thầm giữa notebook 01 và notebook 05.

In [1]:
from pathlib import Path
import shutil
import pandas as pd

In [2]:
OUTPUT_DIR = Path("./outputs")
TABLES_DIR = OUTPUT_DIR / "tables"
CLEAN_DATA_ROOT = Path("../../data/clean")
CLEAN_DATA_ROOT.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(TABLES_DIR / "manifest_filtered.csv")

In [ ]:
if "resolved_sample_rate_hz" not in manifest.columns:
    raise ValueError(
        "manifest_filtered.csv thiếu cột 'resolved_sample_rate_hz' — file này "
        "phải được xây từ io_utils.run_sanity_checks() (notebook 01). Xây lại "
        "manifest ở notebook 01 trước khi chạy tiếp notebook này."
)
    
suspicious = manifest[manifest["warnings"].astype(str).str.contains(
    "NGHI_NGO_SAMPLING_RATE|THOI_LUONG_BAT_THUONG", na=False
)]
if len(suspicious) > 0:
    print(f"[CẦN XÁC MINH THỦ CÔNG] {len(suspicious)} file có fs được SUY LUẬN "
          f"(không phải declared) — kiểm tra lại trước khi tin fs này:")
    display(suspicious[["file_path", "label", "load_hp",
                         "resolved_sample_rate_hz", "warnings"]])
    
missing_fs = manifest[manifest["resolved_sample_rate_hz"].isna()]
if len(missing_fs) > 0:
    print(f"[CẢNH BÁO] {len(missing_fs)} file thiếu resolved_sample_rate_hz — "
          f"sẽ bị loại khỏi manifest_clean, cần kiểm tra thủ công:")
    display(missing_fs[["file_path", "label", "load_hp", "warnings"]])


[CẦN XÁC MINH THỦ CÔNG] 4 file có fs được SUY LUẬN (không phải declared) — kiểm tra lại trước khi tin fs này:


,file_path,label,load_hp,resolved_sample_rate_hz,warnings
36,..\..\data\raw\Normal\100_Normal_3.mat,Normal,3,48000.0,NGHI_NGO_SAMPLING_RATE: n_samples=485643 (12kH...
37,..\..\data\raw\Normal\97_Normal_0.mat,Normal,0,24000.0,NGHI_NGO_SAMPLING_RATE: n_samples=243938 (12kH...
38,..\..\data\raw\Normal\98_Normal_1.mat,Normal,1,48000.0,NGHI_NGO_SAMPLING_RATE: n_samples=483903 (12kH...
39,..\..\data\raw\Normal\99_Normal_2.mat,Normal,2,48000.0,NGHI_NGO_SAMPLING_RATE: n_samples=485063 (12kH...


In [4]:
rows = []
n_skipped_no_fs = 0

for _, row in manifest.iterrows():
    src = Path(row["file_path"])
    if not src.exists():
        print(f"Thiếu file: {src}")
        continue

    resolved_fs = row["resolved_sample_rate_hz"]
    if pd.isna(resolved_fs):
        # KHÔNG fallback âm thầm về 12000Hz — bỏ qua, đã cảnh báo ở cell trên.
        n_skipped_no_fs += 1
        continue

    dst = CLEAN_DATA_ROOT / src.name
    shutil.copy2(src, dst)

    new_row = row.to_dict()
    new_row["file_path"] = str(dst)
    new_row["fs"] = int(resolved_fs)  # <-- nguồn duy nhất: resolved_sample_rate_hz
    rows.append(new_row)

manifest_clean = pd.DataFrame(rows)
manifest_clean.to_csv(TABLES_DIR / "manifest_clean.csv", index=False)

print(f"Đã tổ chức {len(manifest_clean)} file vào {CLEAN_DATA_ROOT}")
if n_skipped_no_fs:
    print(f"[TÓM TẮT] Bỏ qua {n_skipped_no_fs} file do thiếu resolved_sample_rate_hz.")

# Kiểm tra riêng 4 file Normal — để xác nhận fs suy ra có hợp lý không
# (thay vì tin mù dict gõ tay như trước).
normal_check = manifest_clean[manifest_clean["label"] == "Normal"][
    ["file_path", "load_hp", "fs"]
]
print("\nKiểm tra fs suy ra cho các file Normal (so với kỳ vọng CWRU ~48kHz):")
display(normal_check)

manifest_clean[["label", "load_hp", "fs"]]


Đã tổ chức 40 file vào ..\..\data\clean

Kiểm tra fs suy ra cho các file Normal (so với kỳ vọng CWRU ~48kHz):


,file_path,load_hp,fs
36,..\..\data\clean\100_Normal_3.mat,3,48000
37,..\..\data\clean\97_Normal_0.mat,0,24000
38,..\..\data\clean\98_Normal_1.mat,1,48000
39,..\..\data\clean\99_Normal_2.mat,2,48000


,label,load_hp,fs
0,B,0,12000
1,B,1,12000
2,B,2,12000
3,B,3,12000
4,B,0,12000
5,B,1,12000
6,B,2,12000
7,B,3,12000
8,B,0,12000
9,B,1,12000
